# 🌍 WanderAI v2 — Smart Tourism Planner
### Now with Smart Budget Optimizer Agent + Professional Light UI

**What's New in v2:**
- ★ **Smart Budget Optimizer Agent** — enter exact ₹ budget, get 3 tailored plans
- 🎨 **Redesigned professional light UI** — teal + coral + clean white
- 💰 **Number input for budget** — precise INR amount, not just a dropdown
- 📊 **Budget allocation tables** — Frugal / Balanced / Indulgent scenarios

---

### 🤖 10 AI Agents
| # | Agent | Output |
|---|-------|--------|
| 0 | ✈️ Trip Profiler | Budget tier, vibe, visa, best areas |
| ★ | 💡 Smart Budget Optimizer | 3-plan budget table, 20 hacks, daily tracker |
| 1 | 🗺️ Destination Guide | Attractions, hidden gems, visa, language |
| 2 | 🏨 Hotel Advisor | Budget/mid/luxury picks, booking strategy |
| 3 | 🚌 Transport Planner | Flights, local transport, routes, apps |
| 4 | 📅 Itinerary Builder | Day-by-day with times, costs, photo spots |
| 5 | 💰 Expense Tracker | Budget breakdown, currency, free activities |
| 6 | 🍜 Food & Culture | Must-eat dishes, restaurants, festivals |
| 7 | 🛡️ Safety Guide | Scam alerts, emergency contacts, insurance |
| 8 | 🎒 Packing Expert | Weather, packing list, luggage strategy |

---
### ⚙️ Setup — 3 Steps
1. **Cell 1** → Install packages
2. **Cell 2** → Paste free Groq key from https://console.groq.com
3. **Cell 3** → Run — public Gradio URL appears


In [1]:
!pip install -q groq gradio requests
print("✅ Packages installed — ready to plan your perfect trip!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.2 MB/s eta 0:00:00
✅ Packages installed — ready to plan your perfect trip!


In [2]:
GROQ_API_KEY   = "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"   # 🔑 Free at console.groq.com
SERPER_API_KEY = "34cf9bb4a6965f4ad9bcce86f48f5d693048ac09"       # 🔎 Optional — live travel data

from groq import Groq
try:
    _c = Groq(api_key=GROQ_API_KEY)
    _r = _c.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":"Reply: Ready!"}],
        max_tokens=5
    )
    print("✅ Groq API Key VALID — bon voyage! 🌍")
except Exception as e:
    print(f"❌ Error: {e}  →  Get your free key at https://console.groq.com")

❌ Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}  →  Get your free key at https://console.groq.com


In [3]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  WanderAI v2 — Tourism Planner  ·  10 Specialist Agents            ║
# ║  NEW: Smart Budget Optimizer Agent  ·  Professional Light UI        ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, json, re
from dataclasses import dataclass
from typing import Dict, Tuple
import requests
import gradio as gr
from groq import Groq

In [4]:


GROQ_FAST = "llama-3.1-8b-instant"
GROQ_HQ   = "llama-3.3-70b-versatile"
groq_client = Groq(api_key=GROQ_API_KEY)

In [5]:
# ══════════════════════════════════════════════════════════════════════
# HELPER — LLM call
# ══════════════════════════════════════════════════════════════════════
def call_groq(system: str, user: str, max_tokens: int = 900,
              use_hq: bool = False) -> str:
    model = GROQ_HQ if use_hq else GROQ_FAST
    try:
        r = groq_client.chat.completions.create(
            model=model,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            max_tokens=max_tokens, temperature=0.65,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return (f"⚠️ LLM Error: {e}\n\n"
                "Fix: confirm GROQ_API_KEY in Cell 2 is valid, then re-run Cell 3.")

In [6]:

# ══════════════════════════════════════════════════════════════════════
# HELPER — Live search (Serper, optional)
# ══════════════════════════════════════════════════════════════════════
def search_travel(query: str) -> str:
    if not SERPER_API_KEY or SERPER_API_KEY == "YOUR_SERPER_KEY_HERE":
        return "(Live search disabled — add SERPER_API_KEY for real-time data)"
    try:
        r = requests.post(
            "https://google.serper.dev/search",
            json={"q": query, "hl": "en", "num": 4},
            headers={"X-API-KEY": SERPER_API_KEY,
                     "Content-Type": "application/json"},
            timeout=8,
        )
        r.raise_for_status()
        items = r.json().get("organic", [])[:4]
        return "\n".join(f"• {x['title']}: {x.get('snippet','')}" for x in items)
    except Exception as e:
        return f"(Search unavailable: {e})"

In [7]:

# ══════════════════════════════════════════════════════════════════════
# AGENT 0 — Trip Profile Analyzer
# ══════════════════════════════════════════════════════════════════════
@dataclass
class TripProfileAgent:
    role: str = "Trip Profile Analyst"

    def run(self, destination, origin, travel_type, duration,
            group, budget_inr, month, interests) -> Dict:
        system = (
            "You are a senior travel consultant. Analyse the trip and return JSON with keys: "
            "destination_tier (Budget/Mid-range/Luxury), trip_vibe (Adventure/Relaxation/Cultural/Mixed), "
            "daily_budget_usd (integer), best_areas (list of 3 stay zones), "
            "top_3_priorities (list), potential_challenges (list of 2), "
            "ideal_duration (string), visa_required (string for Indian passport). "
            "Return ONLY valid JSON, no markdown."
        )
        user = (
            f"Destination:{destination} | From:{origin} | Type:{travel_type}\n"
            f"Duration:{duration} | Group:{group} | Budget:₹{budget_inr} total\n"
            f"Month:{month} | Interests:{interests}"
        )
        raw = call_groq(system, user, max_tokens=450)
        try:
            s, e = raw.find("{"), raw.rfind("}") + 1
            return json.loads(raw[s:e])
        except Exception:
            return {
                "destination_tier": "Mid-range", "trip_vibe": "Mixed",
                "daily_budget_usd": 80,
                "best_areas": ["City Centre", "Old Town", "Scenic Zone"],
                "top_3_priorities": [travel_type, "Local food", "Sightseeing"],
                "potential_challenges": ["Language barrier", "Peak-season crowds"],
                "ideal_duration": duration, "visa_required": "Check embassy website",
            }

In [8]:

# ══════════════════════════════════════════════════════════════════════
# ★ NEW AGENT — Smart Budget Optimizer
# Takes exact INR budget → builds 3 smart plans + picks the best one
# ══════════════════════════════════════════════════════════════════════
@dataclass
class SmartBudgetOptimizerAgent:
    """
    Agent NEW — Smart Budget Optimizer
    Takes the traveller's exact total budget in INR and intelligently
    allocates it across flights, hotels, food, transport, activities,
    shopping and emergency fund. Generates THREE complete plans
    (Frugal / Balanced / Indulgent) and recommends the best one
    with a detailed allocation table and saving strategies.
    """
    role: str = "Smart Budget Optimizer"

    def run(self, destination: str, origin: str, budget_inr: str,
            duration: str, group: str, travel_type: str,
            month: str, profile: Dict) -> str:
        cost_data = search_travel(
            f"average travel cost {destination} from India per person {duration} 2025"
        )
        system = (
            "You are a certified travel financial planner and budget optimization expert. "
            "The traveller has told you their EXACT total budget in Indian Rupees (₹). "
            "Your job is to make every rupee count. Provide:\n\n"
            "## 1. Budget Reality Check\n"
            "- Is this budget realistic for this destination and duration?\n"
            "- What travel tier does this budget unlock (budget/mid/luxury)?\n"
            "- Honest assessment with encouraging tone\n\n"
            "## 2. Three Smart Budget Plans\n"
            "Create a detailed allocation table for all THREE plans:\n"
            "**Plan A — Frugal Explorer** (stretch every rupee, hostels, street food)\n"
            "**Plan B — Balanced Traveller** ⭐ RECOMMENDED (best value for money)\n"
            "**Plan C — Comfort Seeker** (small luxuries, nicer hotels, restaurants)\n\n"
            "For each plan provide a table with columns: Category | Plan A | Plan B | Plan C\n"
            "Categories: ✈️ Flights | 🏨 Hotel (total) | 🍽️ Food (total) | 🚌 Transport | "
            "🎫 Activities | 🛍️ Shopping | 🆘 Emergency (10%) | 📱 SIM/Misc | 💰 TOTAL\n\n"
            "## 3. Recommended Plan Deep-Dive\n"
            "For the RECOMMENDED plan (usually Plan B unless budget is very tight/generous):\n"
            "- Daily spending breakdown (day-by-day spend target)\n"
            "- Specific hotel tier and example names\n"
            "- Food strategy (which meals to splurge vs save)\n"
            "- Which activities to prioritise given the budget\n"
            "- Transport mode recommendations\n\n"
            "## 4. Top 20 Money-Saving Hacks\n"
            "Destination-specific hacks to save 20-30% without sacrificing experience.\n\n"
            "## 5. Budget Tracking Template\n"
            "Provide a simple daily budget tracking table for this trip.\n\n"
            "## 6. If Budget is Too Tight\n"
            "- Alternative cheaper destinations with similar experience\n"
            "- Ways to reduce duration for same experience\n"
            "- Best season for cheaper prices\n\n"
            "Use ₹ AND USD amounts throughout. Be specific and actionable."
        )
        try:
            budget_num = int(str(budget_inr).replace(",", "").replace("₹", "").strip())
            budget_fmt = f"₹{budget_num:,}"
        except Exception:
            budget_num = 50000
            budget_fmt = f"₹{budget_inr}"

        user = (
            f"Destination: {destination} | Departing from: {origin}\n"
            f"TOTAL BUDGET: {budget_fmt} (for the entire trip)\n"
            f"Duration: {duration} | Group: {group} | Type: {travel_type}\n"
            f"Month: {month} | Trip vibe: {profile.get('trip_vibe','Mixed')}\n"
            f"Destination tier estimate: {profile.get('destination_tier','Mid-range')}\n\n"
            f"Live cost data:\n{cost_data}\n\n"
            "Analyse this budget and generate 3 complete plans with detailed allocation tables."
        )
        return call_groq(system, user, max_tokens=1600, use_hq=True)

In [9]:
# ══════════════════════════════════════════════════════════════════════
# AGENT 1 — Destination Research
# ══════════════════════════════════════════════════════════════════════
@dataclass
class DestinationResearchAgent:
    role: str = "Destination Research Specialist"

    def run(self, destination, origin, travel_type, month, interests, profile) -> str:
        live = search_travel(f"{destination} best places visit travel guide {month} 2025")
        system = (
            "You are a world-class travel writer. Provide:\n"
            "1) Destination overview — why it's special (2 paras)\n"
            "2) Top 10 must-see attractions with one-line descriptions\n"
            "3) 5 hidden gems tourists miss\n"
            "4) Best neighbourhoods to explore (character of each)\n"
            "5) Weather & best time — month-by-month\n"
            "6) Visa & entry for Indian passport holders\n"
            "7) Cultural do's and don'ts (8 tips)\n"
            "8) Essential local language phrases (10)\n"
            "9) Best apps to download for this destination\n"
            "Be vivid and inspiring."
        )
        user = (
            f"Destination:{destination} | From:{origin} | Type:{travel_type}\n"
            f"Month:{month} | Interests:{interests}\n"
            f"Best areas:{', '.join(profile.get('best_areas',[]))}\n\n"
            f"Live data:\n{live}\n\nWrite a comprehensive destination guide."
        )
        return call_groq(system, user, max_tokens=1300, use_hq=True)

In [10]:

# ══════════════════════════════════════════════════════════════════════
# AGENT 2 — Hotel Advisor
# ══════════════════════════════════════════════════════════════════════
@dataclass
class HotelAdvisorAgent:
    role: str = "Hotel & Accommodation Specialist"

    def run(self, destination, budget_inr, group, travel_type, duration, profile) -> str:
        live = search_travel(f"best hotels {destination} {travel_type} review 2025")
        try:
            budget_num = int(str(budget_inr).replace(",","").replace("₹","").strip())
            hotel_budget = f"₹{budget_num:,} total trip"
        except Exception:
            hotel_budget = f"₹{budget_inr} total trip"

        system = (
            "You are a luxury travel concierge. Provide:\n"
            "1) Best zone/area to stay and why\n"
            "2) Budget hotels — 3 picks with price per night in ₹ & USD\n"
            "3) Mid-range hotels — 3 picks with prices\n"
            "4) Luxury hotels/resorts — 3 picks with prices\n"
            "5) Unique stays — boutique, heritage, treehouse, houseboat etc.\n"
            "6) Best booking platforms and timing strategy\n"
            "7) Room selection tips and negotiation advice\n"
            "8) Hidden costs to watch (resort fees, taxes, Wi-Fi)\n"
            "Give real hotel names and price ranges."
        )
        user = (
            f"Destination:{destination} | Budget:{hotel_budget}\n"
            f"Group:{group} | Type:{travel_type} | Duration:{duration}\n"
            f"Best areas:{', '.join(profile.get('best_areas',[]))}\n\n"
            f"Live data:\n{live}\n\nProvide complete accommodation guide."
        )
        return call_groq(system, user, max_tokens=1200, use_hq=True)

In [11]:
# ══════════════════════════════════════════════════════════════════════
# AGENT 3 — Transport & Commute Planner
# ══════════════════════════════════════════════════════════════════════
@dataclass
class CommuteTransportAgent:
    role: str = "Travel & Transport Planner"

    def run(self, destination, origin, budget_inr, duration, group, profile) -> str:
        flights = search_travel(f"cheapest flights {origin} to {destination} 2025")
        local   = search_travel(f"local transport {destination} metro taxi cost 2025")
        system = (
            "You are a transport logistics expert. Provide:\n"
            "1) Flights — options, airlines, fare ranges, booking strategy\n"
            "2) Alternative routes (trains/buses if applicable)\n"
            "3) Airport to city transfer — all options with cost + time\n"
            "4) Local transport: metro, bus, taxi, rideshare, rentals\n"
            "5) Day trips and excursions from the city\n"
            "6) Transport budget estimate for whole trip\n"
            "7) Essential transport apps to download\n"
            "Give costs in ₹ and USD."
        )
        user = (
            f"Destination:{destination} | From:{origin}\n"
            f"Budget:₹{budget_inr} total | Duration:{duration} | Group:{group}\n\n"
            f"Flight data:\n{flights}\n\nLocal transport:\n{local}\n\n"
            "Build a complete transport guide."
        )
        return call_groq(system, user, max_tokens=1200, use_hq=True)

In [12]:
# ══════════════════════════════════════════════════════════════════════
# AGENT 4 — Day-by-Day Itinerary
# ══════════════════════════════════════════════════════════════════════
@dataclass
class ItineraryPlannerAgent:
    role: str = "Itinerary Planning Expert"

    def run(self, destination, duration, travel_type, interests, group, profile) -> str:
        system = (
            "You are an expert travel itinerary planner. Create:\n"
            "1) Pre-trip checklist (5 items)\n"
            "2) Day-by-day schedule:\n"
            "   Each day: Morning (9am-12pm) / Afternoon (12pm-6pm) / Evening (6pm-10pm)\n"
            "   Include: attraction, travel time, entry cost, pro tip\n"
            "3) Time-saving hacks and queue-skipping tips\n"
            "4) Must-see vs can-skip honest ranking\n"
            "5) 5 best photo spots with timing\n"
            "6) Last-day optimisation + airport timing\n"
            "Be specific with times, names, and costs."
        )
        user = (
            f"Destination:{destination} | Duration:{duration}\n"
            f"Type:{travel_type} | Interests:{interests} | Group:{group}\n"
            f"Priorities:{', '.join(profile.get('top_3_priorities',[]))}\n\n"
            "Create a complete practical itinerary."
        )
        return call_groq(system, user, max_tokens=1400, use_hq=True)

In [13]:
# ══════════════════════════════════════════════════════════════════════
# AGENT 5 — Expense & Budget Breakdown
# ══════════════════════════════════════════════════════════════════════
@dataclass
class ExpenseBudgetAgent:
    role: str = "Travel Budget Expert"

    def run(self, destination, origin, budget_inr, duration, group, profile) -> str:
        cost_data = search_travel(f"cost of travel {destination} daily budget 2025")
        try:
            budget_num = int(str(budget_inr).replace(",","").replace("₹","").strip())
            budget_fmt = f"₹{budget_num:,}"
        except Exception:
            budget_fmt = f"₹{budget_inr}"

        system = (
            "You are a travel finance expert. Given the EXACT budget, provide:\n"
            "1) Complete breakdown: flights, hotels, food, transport, activities, shopping, misc\n"
            "2) Daily spend target per person\n"
            "3) Currency guide — exchange rates, best exchange methods\n"
            "4) 15 destination-specific money saving hacks\n"
            "5) 8 free things to do at this destination\n"
            "6) Tourist traps that waste money\n"
            "7) Best payment methods abroad (card, cash, UPI/apps)\n"
            "Show all amounts in ₹ AND USD."
        )
        user = (
            f"Destination:{destination} | From:{origin}\n"
            f"TOTAL BUDGET:{budget_fmt} | Duration:{duration} | Group:{group}\n\n"
            f"Cost data:\n{cost_data}\n\nBuild complete expense breakdown."
        )
        return call_groq(system, user, max_tokens=1200, use_hq=True)

In [14]:

# ══════════════════════════════════════════════════════════════════════
# AGENT 6 — Food & Culture
# ══════════════════════════════════════════════════════════════════════
@dataclass
class FoodCultureAgent:
    role: str = "Food & Cultural Guide"

    def run(self, destination, travel_type, interests, group, profile) -> str:
        food_data = search_travel(f"best food restaurants local cuisine {destination} must eat 2025")
        system = (
            "You are a food journalist and cultural guide. Provide:\n"
            "1) Local cuisine overview\n"
            "2) 15 must-eat dishes with where to find the best version\n"
            "3) Restaurants by budget: street food / casual / fine dining\n"
            "4) Food markets and night markets\n"
            "5) Dietary options (vegetarian, vegan, halal)\n"
            "6) Food safety tips\n"
            "7) Culinary experiences (cooking classes, food tours)\n"
            "8) Local festivals, cultural activities, authentic souvenirs\n"
            "Make the food sound irresistible."
        )
        user = (
            f"Destination:{destination} | Type:{travel_type}\n"
            f"Interests:{interests} | Group:{group}\n\n"
            f"Food data:\n{food_data}\n\nCreate inspiring food and culture guide."
        )
        return call_groq(system, user, max_tokens=1200, use_hq=True)

In [15]:
# ══════════════════════════════════════════════════════════════════════
# AGENT 7 — Safety & Emergency
# ══════════════════════════════════════════════════════════════════════
@dataclass
class SafetyEmergencyAgent:
    role: str = "Travel Safety Specialist"

    def run(self, destination, origin, group, month, profile) -> str:
        safety_data = search_travel(f"{destination} travel safety scams tourists 2025")
        system = (
            "You are a travel security expert. Provide:\n"
            "1) Overall safety rating and context\n"
            "2) Areas/zones to avoid (with reasons)\n"
            "3) Top 10 tourist scams and how to avoid each\n"
            "4) Health preparation: vaccinations, water safety, medical\n"
            "5) Travel insurance guide and recommended providers\n"
            "6) Emergency contacts: police, ambulance, Indian Embassy, tourist helpline\n"
            "7) Digital safety: devices, cards, data\n"
            "8) Emergency action plan if things go wrong\n"
            "Be honest without being alarmist."
        )
        user = (
            f"Destination:{destination} | From:{origin}\n"
            f"Group:{group} | Month:{month}\n\n"
            f"Safety data:\n{safety_data}\n\nProvide safety guide."
        )
        return call_groq(system, user, max_tokens=1100, use_hq=True)

In [16]:

# ══════════════════════════════════════════════════════════════════════
# AGENT 8 — Packing & Weather
# ══════════════════════════════════════════════════════════════════════
@dataclass
class PackingWeatherAgent:
    role: str = "Packing & Weather Expert"

    def run(self, destination, travel_type, duration, group, month, profile) -> str:
        weather = search_travel(f"{destination} weather {month} temperature what to wear")
        system = (
            "You are a packing and climate expert. Provide:\n"
            "1) Weather overview for the travel month\n"
            "2) Complete packing list by category:\n"
            "   Clothing / Footwear / Toiletries / Electronics / Documents / Accessories\n"
            "3) Activity-specific gear for their travel type\n"
            "4) Luggage strategy and weight tips\n"
            "5) 10 packing hacks to save space\n"
            "6) What NOT to pack (10 common mistakes)\n"
            "7) What to buy there vs bring from home\n"
            "Be specific with quantities and fabric types."
        )
        user = (
            f"Destination:{destination} | Month:{month}\n"
            f"Type:{travel_type} | Duration:{duration} | Group:{group}\n\n"
            f"Weather data:\n{weather}\n\nCreate personalised packing list."
        )
        return call_groq(system, user, max_tokens=1100, use_hq=True)

In [17]:
# ══════════════════════════════════════════════════════════════════════
# MASTER ORCHESTRATOR
# ══════════════════════════════════════════════════════════════════════
def tourism_planner(
    destination, origin, travel_type, duration,
    group, budget_inr, month, interests
):
    if not all(s.strip() for s in [destination, origin]):
        m = "⚠️ Please enter your Destination and Origin city to begin planning!"
        return m, m, m, m, m, m, m, m, m, m

    # Sanitise
    dest    = destination.strip()
    orig    = origin.strip()
    inter   = interests.strip() or "sightseeing, food, culture"
    bud_str = str(budget_inr).replace(",","").replace("₹","").strip() or "50000"

    # ── Profile (Agent 0) ────────────────────────────────────
    profile = TripProfileAgent().run(dest, orig, travel_type, duration,
                                      group, bud_str, month, inter)
    tier  = profile.get("destination_tier","Mid-range")
    vibe  = profile.get("trip_vibe","Mixed")
    daily = profile.get("daily_budget_usd", 80)
    visa  = profile.get("visa_required","Check embassy")

    tier_icon  = {"Budget":"🟢","Mid-range":"🟡","Luxury":"💎"}.get(tier,"🟡")
    vibe_icon  = {"Adventure":"🧗","Relaxation":"🌅","Cultural":"🏛️","Mixed":"🎭"}.get(vibe,"🎭")
    try:
        bud_display = f"₹{int(bud_str):,}"
    except Exception:
        bud_display = f"₹{budget_inr}"

    summary_md = f"""## ✈️ Your Trip at a Glance

| Field | Details |
|-------|---------|
| **Destination** | 🌍 {dest} |
| **Departing From** | 🏠 {orig} |
| **Travel Type** | {travel_type} |
| **Duration** | 🗓️ {duration} |
| **Group** | 👥 {group} |
| **Travel Month** | 📅 {month} |
| **Your Budget** | 💵 {bud_display} total |
| **Budget Tier** | {tier_icon} {tier} |
| **Trip Vibe** | {vibe_icon} {vibe} |
| **Est. Daily/Person** | 💰 ~${daily} USD/day |
| **Visa (Indian 🇮🇳)** | 🛂 {visa} |

### 🎯 Top Priorities for This Trip
{chr(10).join(f"- **{p}**" for p in profile.get("top_3_priorities", [travel_type, "Local food", "Sightseeing"]))}

### 📍 Best Zones to Base Yourself
{chr(10).join(f"- {a}" for a in profile.get("best_areas", []))}

### ⚠️ Things to Plan For
{chr(10).join(f"- {c}" for c in profile.get("potential_challenges", []))}

### 💡 Ideal Trip Length
> **{profile.get('ideal_duration', duration)}** — based on your travel style and interests

---
*All 9 specialist reports are ready in the tabs above. Explore each one for your complete travel plan. Bon voyage! 🌏*"""

    # ── Run all specialist agents ────────────────────────────
    budget_md  = "## 💡 Smart Budget Optimizer\n\n" + SmartBudgetOptimizerAgent().run(dest, orig, bud_str, duration, group, travel_type, month, profile)
    dest_md    = "## 🗺️ Destination Guide\n\n"        + DestinationResearchAgent().run(dest, orig, travel_type, month, inter, profile)
    hotel_md   = "## 🏨 Hotels & Accommodation\n\n"   + HotelAdvisorAgent().run(dest, bud_str, group, travel_type, duration, profile)
    comm_md    = "## 🚌 Transport & Commute\n\n"       + CommuteTransportAgent().run(dest, orig, bud_str, duration, group, profile)
    itin_md    = "## 📅 Day-by-Day Itinerary\n\n"      + ItineraryPlannerAgent().run(dest, duration, travel_type, inter, group, profile)
    exp_md     = "## 💰 Expenses Breakdown\n\n"        + ExpenseBudgetAgent().run(dest, orig, bud_str, duration, group, profile)
    food_md    = "## 🍜 Food & Culture\n\n"             + FoodCultureAgent().run(dest, travel_type, inter, group, profile)
    safe_md    = "## 🛡️ Safety & Emergency\n\n"        + SafetyEmergencyAgent().run(dest, orig, group, month, profile)
    pack_md    = "## 🎒 Packing & Weather\n\n"          + PackingWeatherAgent().run(dest, travel_type, duration, group, month, profile)

    return summary_md, budget_md, dest_md, hotel_md, comm_md, itin_md, exp_md, food_md, safe_md, pack_md

In [18]:
# ══════════════════════════════════════════════════════════════════════
# CSS — Professional Light Theme
# Palette: pure white + warm pearl + deep teal + coral accent
# Fonts: DM Serif Display (elegant) + Plus Jakarta Sans (modern clean)
# ══════════════════════════════════════════════════════════════════════
CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=Plus+Jakarta+Sans:wght@300;400;500;600;700;800&display=swap');

:root {
  --white:      #FFFFFF;
  --pearl:      #F7F4EF;
  --pearl2:     #EEE9E0;
  --teal:       #0E7490;
  --teal-lt:    #0EA5C9;
  --teal-dim:   #0C5F78;
  --teal-pale:  #E0F2FE;
  --coral:      #E05A2B;
  --coral-lt:   #F47548;
  --coral-pale: #FEF0EA;
  --green:      #059669;
  --green-pale: #ECFDF5;
  --amber:      #B45309;
  --amber-pale: #FFFBEB;
  --slate:      #334155;
  --slate-md:   #64748B;
  --slate-lt:   #94A3B8;
  --border:     #E2E8F0;
  --shadow-sm:  0 1px 3px rgba(0,0,0,0.08), 0 1px 2px rgba(0,0,0,0.05);
  --shadow-md:  0 4px 16px rgba(0,0,0,0.08), 0 2px 6px rgba(0,0,0,0.05);
  --shadow-lg:  0 10px 40px rgba(0,0,0,0.1), 0 4px 12px rgba(0,0,0,0.06);
  --radius:     12px;
}

/* ── Reset & Base ─────────────────────────────────────────────── */
* { box-sizing: border-box; }
.gradio-container {
  background: var(--pearl) !important;
  font-family: 'Plus Jakarta Sans', sans-serif !important;
  min-height: 100vh;
  position: relative;
}

/* ── Top accent bar ───────────────────────────────────────────── */
.top-bar {
  height: 4px;
  background: linear-gradient(90deg,
    var(--teal) 0%, var(--teal-lt) 30%,
    var(--coral) 60%, var(--coral-lt) 100%);
  margin-bottom: 0;
}

/* ── Header card ──────────────────────────────────────────────── */
.header-card {
  background: var(--white);
  border-bottom: 1px solid var(--border);
  padding: 32px 24px 24px;
  margin-bottom: 24px;
  box-shadow: var(--shadow-sm);
}
.header-inner {
  max-width: 860px;
  margin: 0 auto;
  text-align: center;
}
.brand-line {
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 14px;
  margin-bottom: 10px;
}
.brand-icon {
  width: 52px; height: 52px;
  background: linear-gradient(135deg, var(--teal), var(--teal-lt));
  border-radius: 14px;
  display: flex; align-items: center; justify-content: center;
  font-size: 1.6rem;
  box-shadow: var(--shadow-md);
}
#wa-title {
  font-family: 'DM Serif Display', serif !important;
  font-size: 2.6rem !important;
  font-weight: 400 !important;
  color: var(--slate) !important;
  margin: 0 !important;
  letter-spacing: -0.5px;
  line-height: 1;
}
#wa-title span { color: var(--teal); }
.tagline {
  font-size: 0.9rem;
  color: var(--slate-md);
  letter-spacing: 0.3px;
  margin: 6px 0 16px;
  font-weight: 400;
}
.tagline strong { color: var(--teal-dim); font-weight: 600; }
/* Agent chips in header */
.chip-row {
  display: flex;
  gap: 7px;
  justify-content: center;
  flex-wrap: wrap;
}
.chip {
  background: var(--teal-pale);
  color: var(--teal-dim);
  border: 1px solid rgba(14,116,144,0.2);
  border-radius: 20px;
  padding: 4px 12px;
  font-size: 0.72rem;
  font-weight: 600;
  letter-spacing: 0.2px;
}
.chip.new-chip {
  background: linear-gradient(135deg, var(--coral-pale), #FFF0E6);
  color: var(--coral);
  border-color: rgba(224,90,43,0.25);
  font-weight: 700;
}

/* ── Section headings ─────────────────────────────────────────── */
.form-section {
  background: var(--white);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  padding: 20px 20px 16px;
  margin-bottom: 16px;
  box-shadow: var(--shadow-sm);
}
.sec-head {
  font-family: 'Plus Jakarta Sans', sans-serif;
  font-size: 0.72rem;
  font-weight: 700;
  color: var(--teal-dim);
  text-transform: uppercase;
  letter-spacing: 2px;
  margin: 0 0 14px;
  padding-bottom: 10px;
  border-bottom: 2px solid var(--teal-pale);
  display: flex;
  align-items: center;
  gap: 8px;
}

/* ── Budget highlight box ─────────────────────────────────────── */
.budget-box {
  background: linear-gradient(135deg, var(--coral-pale), #FFF5EF);
  border: 1.5px solid rgba(224,90,43,0.2);
  border-radius: var(--radius);
  padding: 20px;
  margin-bottom: 16px;
  box-shadow: var(--shadow-sm);
}
.budget-label {
  font-size: 0.7rem;
  font-weight: 700;
  color: var(--coral);
  text-transform: uppercase;
  letter-spacing: 2px;
  margin-bottom: 10px;
  display: flex;
  align-items: center;
  gap: 6px;
}
.budget-label span {
  background: var(--coral);
  color: white;
  border-radius: 20px;
  padding: 2px 8px;
  font-size: 0.62rem;
  letter-spacing: 1px;
}

/* ══════════════════════════════════════════════════════════════
   INPUT FIX — all inputs must have dark text on white bg
══════════════════════════════════════════════════════════════ */
.gradio-container input[type="text"],
.gradio-container input[type="number"],
.gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container .block input,
.gradio-container .block textarea,
input, textarea, select {
  background-color: #FFFFFF !important;
  color: #1E293B !important;
  border: 1.5px solid var(--border) !important;
  border-radius: 8px !important;
  font-family: 'Plus Jakarta Sans', sans-serif !important;
  font-size: 0.88rem !important;
  font-weight: 400 !important;
  padding: 10px 14px !important;
  caret-color: var(--teal) !important;
  box-shadow: var(--shadow-sm) !important;
  transition: border-color 0.2s, box-shadow 0.2s, background 0.15s;
}
.gradio-container input:focus,
.gradio-container textarea:focus,
input:focus, textarea:focus {
  border-color: var(--teal) !important;
  box-shadow: 0 0 0 3px rgba(14,116,144,0.12) !important;
  outline: none !important;
  color: #1E293B !important;
  background-color: #FAFEFF !important;
}
input::placeholder, textarea::placeholder,
.gradio-container input::placeholder,
.gradio-container textarea::placeholder {
  color: #B0BEC5 !important;
  opacity: 1 !important;
}

/* ── Labels ───────────────────────────────────────────────────── */
.gradio-container label > span,
.gradio-container .label-wrap > span {
  font-family: 'Plus Jakarta Sans', sans-serif !important;
  font-size: 0.72rem !important;
  font-weight: 700 !important;
  color: var(--slate) !important;
  text-transform: uppercase !important;
  letter-spacing: 0.8px !important;
}

/* ── Number input (budget slider area) ─────────────────────────── */
.gradio-container input[type="number"] {
  background: #FFFFFF !important;
  color: #1E293B !important;
  font-size: 1.05rem !important;
  font-weight: 600 !important;
  border-color: rgba(224,90,43,0.3) !important;
}
.gradio-container input[type="number"]:focus {
  border-color: var(--coral) !important;
  box-shadow: 0 0 0 3px rgba(224,90,43,0.1) !important;
}

/* ── CTA Button ───────────────────────────────────────────────── */
#plan-btn {
  background: linear-gradient(135deg, var(--teal), var(--teal-lt)) !important;
  color: #FFFFFF !important;
  font-family: 'Plus Jakarta Sans', sans-serif !important;
  font-weight: 700 !important;
  font-size: 0.98rem !important;
  letter-spacing: 0.5px !important;
  border: none !important;
  border-radius: 10px !important;
  padding: 14px 36px !important;
  width: 100% !important;
  box-shadow: 0 4px 20px rgba(14,116,144,0.35) !important;
  transition: transform 0.15s, box-shadow 0.15s !important;
  cursor: pointer !important;
  position: relative;
  overflow: hidden;
}
#plan-btn:hover {
  transform: translateY(-2px) !important;
  box-shadow: 0 8px 30px rgba(14,116,144,0.45) !important;
}
#plan-btn::after {
  content: '';
  position: absolute; top:0; left:-100%; width:60%; height:100%;
  background: linear-gradient(90deg, transparent, rgba(255,255,255,0.15), transparent);
  animation: btn-shim 2.2s ease-in-out infinite;
}
@keyframes btn-shim { 0%{left:-100%} 100%{left:200%} }

/* ── Tab navigation ───────────────────────────────────────────── */
.tab-nav {
  background: var(--white) !important;
  border-bottom: 2px solid var(--border) !important;
  padding: 0 4px !important;
}
.tab-nav button {
  font-family: 'Plus Jakarta Sans', sans-serif !important;
  font-size: 0.76rem !important;
  font-weight: 600 !important;
  color: var(--slate-md) !important;
  background: transparent !important;
  border: none !important;
  border-bottom: 2px solid transparent !important;
  margin-bottom: -2px !important;
  padding: 11px 14px !important;
  letter-spacing: 0.2px;
  transition: color 0.2s, border-color 0.2s !important;
}
.tab-nav button:hover {
  color: var(--teal) !important;
}
.tab-nav button.selected,
.tab-nav button[aria-selected="true"] {
  color: var(--teal) !important;
  border-bottom-color: var(--teal) !important;
  background: transparent !important;
}

/* ── Output tab content area ──────────────────────────────────── */
.tab-content-wrap {
  background: var(--white);
  border: 1px solid var(--border);
  border-radius: 0 0 var(--radius) var(--radius);
  box-shadow: var(--shadow-sm);
}

/* ── Markdown output ──────────────────────────────────────────── */
.out-md, .out-md * {
  font-family: 'Plus Jakarta Sans', sans-serif !important;
  color: var(--slate) !important;
  line-height: 1.8 !important;
}
.out-md h2 {
  font-family: 'DM Serif Display', serif !important;
  color: var(--teal-dim) !important;
  font-size: 1.5rem !important;
  font-weight: 400 !important;
  border-bottom: 2px solid var(--teal-pale);
  padding-bottom: 10px;
  margin-bottom: 18px;
  letter-spacing: -0.3px;
}
.out-md h3 {
  font-family: 'DM Serif Display', serif !important;
  color: var(--coral) !important;
  font-size: 1.1rem !important;
  font-weight: 400 !important;
  margin: 20px 0 8px !important;
}
.out-md h4 {
  font-family: 'Plus Jakarta Sans', sans-serif !important;
  color: var(--slate) !important;
  font-size: 0.82rem !important;
  font-weight: 700 !important;
  text-transform: uppercase;
  letter-spacing: 1px;
  margin: 16px 0 7px !important;
}
.out-md strong { color: var(--teal-dim) !important; font-weight: 700 !important; }
.out-md em     { color: var(--slate-md) !important; font-style: italic; }
.out-md a      { color: var(--teal) !important; text-decoration: underline; }
.out-md blockquote {
  border-left: 4px solid var(--teal-lt);
  background: var(--teal-pale);
  border-radius: 0 8px 8px 0;
  padding: 12px 18px;
  margin: 14px 0;
  color: var(--teal-dim) !important;
  font-style: italic;
}
.out-md code {
  background: var(--teal-pale) !important;
  color: var(--teal-dim) !important;
  border: 1px solid rgba(14,116,144,0.15);
  border-radius: 4px;
  padding: 2px 7px;
  font-size: 0.84rem;
  font-family: 'Courier New', monospace !important;
}
/* Table styling */
.out-md table {
  width: 100%;
  border-collapse: collapse;
  margin: 16px 0;
  border-radius: 8px;
  overflow: hidden;
  box-shadow: var(--shadow-sm);
}
.out-md th {
  background: var(--teal) !important;
  color: #FFFFFF !important;
  font-size: 0.75rem !important;
  text-transform: uppercase;
  letter-spacing: 0.8px;
  padding: 10px 14px;
  font-weight: 600 !important;
  text-align: left;
  border: none !important;
}
.out-md td {
  color: var(--slate) !important;
  padding: 9px 14px;
  border: none !important;
  border-bottom: 1px solid var(--border) !important;
  font-size: 0.88rem;
}
.out-md tr:nth-child(even) td { background: var(--pearl) !important; }
.out-md tr:hover td { background: var(--teal-pale) !important; }
.out-md ul { padding-left: 18px; }
.out-md ul li { margin: 5px 0; }
.out-md ul li::marker { color: var(--teal) !important; }
.out-md ol li::marker { color: var(--coral) !important; font-weight: 700; }
.out-md hr {
  border: none !important;
  border-top: 2px solid var(--border) !important;
  margin: 18px 0;
}

/* ── Examples table ───────────────────────────────────────────── */
.examples-holder, .gr-samples-table {
  background: var(--white);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  overflow: hidden;
  box-shadow: var(--shadow-sm);
}
.examples-holder td, .gr-samples-table td {
  color: var(--slate) !important;
  background: var(--white) !important;
  font-size: 0.8rem !important;
  border-bottom: 1px solid var(--border) !important;
  padding: 8px 12px !important;
  font-family: 'Plus Jakarta Sans', sans-serif !important;
}
.examples-holder tr:hover td {
  background: var(--teal-pale) !important;
  cursor: pointer;
  color: var(--teal-dim) !important;
}
.examples-holder th, .gr-samples-table th {
  background: var(--pearl2) !important;
  color: var(--slate-md) !important;
  font-size: 0.7rem !important;
  text-transform: uppercase;
  letter-spacing: 1px;
  padding: 9px 12px !important;
  font-weight: 700 !important;
  border-bottom: 1px solid var(--border) !important;
}

/* ── Footer ───────────────────────────────────────────────────── */
.wa-footer {
  background: var(--white);
  border-top: 1px solid var(--border);
  padding: 20px 24px;
  margin-top: 28px;
  display: flex;
  justify-content: space-between;
  align-items: center;
  flex-wrap: wrap;
  gap: 10px;
  box-shadow: 0 -2px 8px rgba(0,0,0,0.04);
}
.footer-brand {
  font-family: 'DM Serif Display', serif;
  font-size: 1.2rem;
  color: var(--teal);
  letter-spacing: -0.3px;
}
.footer-links {
  font-size: 0.75rem;
  color: var(--slate-lt);
  letter-spacing: 0.5px;
}
.footer-badge {
  background: var(--teal-pale);
  color: var(--teal-dim);
  border: 1px solid rgba(14,116,144,0.2);
  border-radius: 20px;
  padding: 4px 12px;
  font-size: 0.7rem;
  font-weight: 600;
}

/* ── Scrollbar ────────────────────────────────────────────────── */
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: var(--pearl); }
::-webkit-scrollbar-thumb { background: #CBD5E1; border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--teal-lt); }
"""

# ══════════════════════════════════════════════════════════════════════
# DATA
# ══════════════════════════════════════════════════════════════════════
EXAMPLES = [
    ["Bali, Indonesia",    "Mumbai",    "🏖️ Beach & Coastal",       "7 days",          "Couple",       "75000",  "July",      "Surfing, temples, rice terraces, nightlife"],
    ["Paris, France",      "Delhi",     "🏛️ Historical & Cultural", "10 days",         "Couple",       "250000", "September", "Art, architecture, fine dining, fashion"],
    ["Manali, India",      "Bangalore", "🏔️ Mountains & Trekking",  "5 days",          "Friends (3-5)","18000",  "June",      "Trekking, snow, camping, adventure"],
    ["Bangkok, Thailand",  "Chennai",   "🍽️ Food & Culinary Tour",  "6 days",          "Solo",         "35000",  "November",  "Street food, night markets, temples"],
    ["Rajasthan, India",   "Kolkata",   "🏛️ Historical & Cultural", "8 days",          "Family (2+2)", "55000",  "February",  "Forts, palaces, camel safari, heritage"],
    ["Maldives",           "Pune",      "💑 Honeymoon & Romantic",  "5 days",          "Couple",       "180000", "March",     "Overwater villa, snorkelling, sunset cruise"],
    ["Ladakh, India",      "Delhi",     "🎒 Budget Backpacking",    "10 days",         "Friends (3-5)","22000",  "August",    "Monasteries, Pangong lake, Nubra valley"],
    ["Switzerland",        "Hyderabad", "🏔️ Mountains & Trekking",  "12 days",         "Family (2+2)", "400000", "December",  "Alps, skiing, chocolate, scenic trains"],
]

TRAVEL_TYPES = [
    "🏖️ Beach & Coastal", "🏔️ Mountains & Trekking",
    "🏛️ Historical & Cultural", "🌆 City & Urban Explorer",
    "🌿 Nature & Wildlife", "🍽️ Food & Culinary Tour",
    "💑 Honeymoon & Romantic", "👨‍👩‍👧 Family Vacation",
    "🎒 Budget Backpacking", "💼 Business + Leisure",
    "🧘 Wellness & Retreat", "🎉 Adventure & Thrill",
]
DURATION_OPTIONS = [
    "2-3 days (Weekend)", "4-5 days", "6-7 days (One week)",
    "8-10 days", "10-14 days (2 weeks)", "15-21 days", "1 month+",
]
GROUP_OPTIONS = [
    "Solo", "Couple", "Friends (3-5)", "Family (2+2)",
    "Large Group (6+)", "Backpacker solo",
]
MONTHS = ["January","February","March","April","May","June",
          "July","August","September","October","November","December"]


# ══════════════════════════════════════════════════════════════════════
# GRADIO UI
# ══════════════════════════════════════════════════════════════════════
def build_ui() -> gr.Blocks:
    with gr.Blocks(title="WanderAI — Smart Tourism Planner") as demo:

        # ── Top accent bar ────────────────────────────────────
        gr.HTML('<div class="top-bar"></div>')

        # ── Header ────────────────────────────────────────────
        gr.HTML("""
        <div class="header-card">
          <div class="header-inner">
            <div class="brand-line">
              <div class="brand-icon">🌍</div>
              <h1 id="wa-title">Wander<span>AI</span></h1>
            </div>
            <p class="tagline">
              Your intelligent travel companion &nbsp;·&nbsp;
              <strong>10 AI Agents</strong> &nbsp;·&nbsp;
              Hotels · Transport · Budget · Itinerary · Food · Safety · Packing
            </p>
            <div class="chip-row">
              <span class="chip">✈️ Trip Profiler</span>
              <span class="chip new-chip">💡 Smart Budget ★ NEW</span>
              <span class="chip">🗺️ Destination</span>
              <span class="chip">🏨 Hotels</span>
              <span class="chip">🚌 Transport</span>
              <span class="chip">📅 Itinerary</span>
              <span class="chip">💰 Expenses</span>
              <span class="chip">🍜 Food</span>
              <span class="chip">🛡️ Safety</span>
              <span class="chip">🎒 Packing</span>
            </div>
          </div>
        </div>""")

        # ── WHERE & WHEN ──────────────────────────────────────
        gr.HTML("""<div class="form-section">
          <div class="sec-head">🌍 &nbsp; Destination & Timing</div>""")
        with gr.Row():
            dest_in  = gr.Textbox(label="✈️  Destination",
                                  placeholder="e.g. Bali, Paris, Manali, Tokyo, Rajasthan …")
            orig_in  = gr.Textbox(label="🏠  Departing From",
                                  placeholder="e.g. Mumbai, Delhi, Bangalore …")
            month_in = gr.Dropdown(label="📅  Travel Month",
                                   choices=MONTHS, value="October")
        gr.HTML("</div>")

        # ── TRIP STYLE ────────────────────────────────────────
        gr.HTML("""<div class="form-section">
          <div class="sec-head">🎒 &nbsp; Trip Style & Group</div>""")
        with gr.Row():
            ttype_in = gr.Dropdown(label="🌈  Travel Type",
                                   choices=TRAVEL_TYPES,
                                   value="🏛️ Historical & Cultural")
            group_in = gr.Dropdown(label="👥  Travelling With",
                                   choices=GROUP_OPTIONS, value="Couple")
            dur_in   = gr.Dropdown(label="🗓️  Duration",
                                   choices=DURATION_OPTIONS,
                                   value="6-7 days (One week)")
        gr.HTML("</div>")

        # ── ★ BUDGET INPUT (NEW) ──────────────────────────────
        gr.HTML("""<div class="budget-box">
          <div class="budget-label">💡 &nbsp; Your Total Trip Budget &nbsp;
            <span>★ NEW — Smart Budget Optimizer</span>
          </div>""")
        with gr.Row():
            budget_in = gr.Number(
                label="💰  Total Budget (₹ INR) — Enter your complete trip budget",
                value=50000,
                minimum=5000,
                maximum=10000000,
                step=1000,
                info="Enter your total budget for the entire trip including flights, hotel, food, activities & shopping.",
            )
            interests_in = gr.Textbox(
                label="🎯  Your Interests & Special Requests",
                placeholder="e.g. temples, street food, hiking, photography, nightlife, local markets, luxury spa …",
                lines=2,
            )
        gr.HTML("</div>")

        # ── LAUNCH BUTTON ─────────────────────────────────────
        with gr.Row():
            with gr.Column(scale=1): gr.HTML("")
            with gr.Column(scale=3):
                plan_btn = gr.Button(
                    "🌍   Plan My Perfect Trip",
                    variant="primary",
                    elem_id="plan-btn",
                )
            with gr.Column(scale=1): gr.HTML("")

        # ── Sample trips ──────────────────────────────────────
        gr.Examples(
            examples=EXAMPLES,
            inputs=[dest_in, orig_in, ttype_in, dur_in,
                    group_in, budget_in, month_in, interests_in],
            label="🌏  Sample Trips — click any row to auto-fill all fields",
        )

        # ── Divider ───────────────────────────────────────────
        gr.HTML('<div class="top-bar" style="margin:24px 0 18px;opacity:0.4;"></div>')

        # ── OUTPUT TABS ───────────────────────────────────────
        gr.HTML("""<div class="sec-head" style="margin-bottom:14px;padding:0 4px;">
          📋 &nbsp; Your Complete Travel Plan</div>""")

        with gr.Tabs():
            with gr.TabItem("✈️ Summary"):
                sum_out  = gr.Markdown(
                    value="*Your trip overview will appear here after planning …*",
                    elem_classes="out-md")
            with gr.TabItem("💡 Smart Budget ★"):
                bud_out  = gr.Markdown(
                    value="*3-plan budget analysis (Frugal / Balanced / Indulgent) will appear here …*",
                    elem_classes="out-md")
            with gr.TabItem("🗺️ Destination"):
                dest_out = gr.Markdown(
                    value="*Destination guide, attractions & hidden gems …*",
                    elem_classes="out-md")
            with gr.TabItem("🏨 Hotels"):
                hot_out  = gr.Markdown(
                    value="*Hotel recommendations across all budgets …*",
                    elem_classes="out-md")
            with gr.TabItem("🚌 Transport"):
                com_out  = gr.Markdown(
                    value="*Flights, local transport & commute guide …*",
                    elem_classes="out-md")
            with gr.TabItem("📅 Itinerary"):
                itin_out = gr.Markdown(
                    value="*Day-by-day itinerary with times and tips …*",
                    elem_classes="out-md")
            with gr.TabItem("💰 Expenses"):
                exp_out  = gr.Markdown(
                    value="*Detailed expense breakdown & money tips …*",
                    elem_classes="out-md")
            with gr.TabItem("🍜 Food & Culture"):
                food_out = gr.Markdown(
                    value="*Local cuisine, restaurants & cultural guide …*",
                    elem_classes="out-md")
            with gr.TabItem("🛡️ Safety"):
                safe_out = gr.Markdown(
                    value="*Safety tips, scam alerts & emergency contacts …*",
                    elem_classes="out-md")
            with gr.TabItem("🎒 Packing"):
                pack_out = gr.Markdown(
                    value="*Weather forecast & personalised packing list …*",
                    elem_classes="out-md")

        # ── Footer ────────────────────────────────────────────
        gr.HTML("""
        <div class="wa-footer">
          <span class="footer-brand">WanderAI</span>
          <span class="footer-links">
            Groq LLaMA 3 Open-Source &nbsp;·&nbsp;
            10 AI Agents &nbsp;·&nbsp;
            For planning purposes only — always verify bookings independently
          </span>
          <span class="footer-badge">✈️ Bon Voyage!</span>
        </div>""")

        # ── Wire up ───────────────────────────────────────────
        plan_btn.click(
            fn=tourism_planner,
            inputs=[dest_in, orig_in, ttype_in, dur_in,
                    group_in, budget_in, month_in, interests_in],
            outputs=[sum_out, bud_out, dest_out, hot_out, com_out,
                     itin_out, exp_out, food_out, safe_out, pack_out],
            show_progress="full",
        )

    return demo



In [19]:

# ══════════════════════════════════════════════════════════════════════
# LAUNCH
# ══════════════════════════════════════════════════════════════════════
print("\n" + "═"*62)
print("  🌍  WanderAI v2 — Smart Tourism Planner")
print(f"  Fast : Groq {GROQ_FAST}")
print(f"  HQ   : Groq {GROQ_HQ}")
print("  NEW  : Smart Budget Optimizer Agent (3-plan analysis)")
print("  UI   : Professional light theme — teal + coral + white")
print("  Tabs : 10 output sections")
print("  ➜  Public share link below ...")
print("═"*62 + "\n")

build_ui().launch(share=True, show_error=True, debug=False)



══════════════════════════════════════════════════════════════
  🌍  WanderAI v2 — Smart Tourism Planner
  Fast : Groq llama-3.1-8b-instant
  HQ   : Groq llama-3.3-70b-versatile
  NEW  : Smart Budget Optimizer Agent (3-plan analysis)
  UI   : Professional light theme — teal + coral + white
  Tabs : 10 output sections
  ➜  Public share link below ...
══════════════════════════════════════════════════════════════



/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: 7 days or set allow_custom_value=True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: 10 days or set allow_custom_value=True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: 5 days or set allow_custom_value=True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: 6 days or set allow_custom_

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6ea63baeff4b80db87.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 🔑 Free Groq API Key
1. Visit https://console.groq.com
2. Sign up free → API Keys → Create Key
3. Paste in Cell 2 — test confirms it works

## ★ How Smart Budget Optimizer Works
1. Enter your **exact total budget in ₹ INR**
2. Agent analyses destination costs in real-time
3. Generates **3 complete plans**: Frugal / Balanced / Indulgent
4. Recommends the **best plan** for your budget
5. Provides **daily spend targets** and **20 saving hacks**
6. If budget is tight → suggests cheaper alternatives

## ⚠️ Disclaimer
AI-generated travel information for planning purposes only.
Always verify hotel prices, flight fares, visa requirements
and safety advisories through official sources before booking.
